# Panel del pipeline LMI-Net (Partes A–F)

Notebook de control del pipeline nuevo. El motor vive en los paquetes; este notebook solo
**orquesta**:

1. **Verificación** — tests obligatorios (Parte A/B/C) + speedup del backward matrix-free.
2. **Barrido por etapas** — smoke `E0` de punta a punta y cómo lanzar `E1`–`E4` (paquete `barrido`).
3. **OOD / profesor** — bancos pole-shift y politopo del profesor con la instrumentación del
   mecanismo de fallo (Parte E).
4. **Figuras** — genera las figuras (entrenamiento + P1–P5) desde los CSV/parquet crudos
   (Parte F), sin recomputar.

Módulos: `red/` (solvers unificados + backward implícito matrix-free), `entrenamiento/training.py`
(6 pérdidas), `barrido/` (barrido, OOD, figuras).

In [ ]:
import os, sys, warnings, time
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name != "Red" and (ROOT / "Red").exists():   # por si el cwd es la raíz del repo
    ROOT = ROOT / "Red"
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
torch.set_default_dtype(torch.float64); torch.set_num_threads(6)
plt.rcParams.update({"font.family": "serif", "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})
print("cwd:", Path.cwd(), "| torch", torch.__version__)

## 1. Verificación (Partes A, B, C)

- **A** `test_solver_equivalence`: unrolling e implícita dan el **mismo** `(Q, Y)` a
  convergencia (salida unificada: ambas aterrizan con la proyección final Π_{C1}).
- **B** `test_implicit_grad_matches_jacrev`: el VJP **matrix-free** coincide con el `jacrev`
  de referencia (~1e-6).
- **C** `test_losses_smoke`: las 6 pérdidas corren sin `NaN`/`Inf`.
- **speedup**: backward matrix-free vs `jacrev` en `n_x=5, N=5`.

> Si el gradiente matrix-free NO baja de 1e-6, suele ser la convergencia del punto fijo
> (Neumann) en modos casi-tangentes C1/C2; el fallback documentado es GMRES en
> `red.backprop_implicit._adjoint_solve`.

In [ ]:
from tests.test_solver_equivalence import test_solver_equivalence
from tests.test_implicit_grad_matches_jacrev import test_implicit_grad_matches_jacrev
from tests.test_losses_smoke import test_losses_smoke
from tests import bench_implicit_backward as bench

test_solver_equivalence();            print("A OK: salida unificada")
test_implicit_grad_matches_jacrev();  print("B OK: VJP matrix-free == jacrev")
test_losses_smoke();                  print("C OK: 6 pérdidas sin NaN/Inf")

print("\n-- speedup backward matrix-free vs jacrev (n_x=5, N=5) --")
bench.main()

## 2. Barrido por etapas (Parte D)

Una corrida = una config completa (`RunConfig`). Épocas `{50,100,200,400}` y
`dr_eval {100…8000}` son **checkpoints**: no multiplican el nº de entrenamientos. Todo se
guarda **crudo** (una fila por sistema × anillo × época × dr_eval); la **semilla es columna**
y nunca se promedia en crudo.

Abajo corremos **E0** (smoke rápido) de punta a punta y **secuencial** (en notebook evitamos
el multiproceso). Las etapas grandes se lanzan desde la terminal (celda siguiente).

In [ ]:
from barrido.config import build_stage
from barrido.run import run_one

OUT_NB = ROOT / "analisis" / "resultados" / "barrido" / "E0_nb"
cfgs = build_stage("E0")
print(f"E0: {len(cfgs)} corridas -> {OUT_NB}")
res_e0 = [run_one(c, OUT_NB) for c in cfgs]     # secuencial, sin Pool (Windows/Jupyter friendly)
pd.DataFrame(res_e0)

### Lanzar las etapas grandes (terminal, multiproceso)

```bash
python -m barrido.driver --stage E1 --workers 16      # factorial núcleo (el bloque grande)
python -m barrido.driver --stage E2 --workers 16      # confirmación de orden n_x
python -m barrido.driver --stage E3 --workers 16      # curva de datos (tamaño)
python -m barrido.driver --stage E4 --workers 16      # backprop + escalera de arquitectura
python -m barrido.driver --stage E1 --cvxpy           # + techo CVXPY en A1/A2 (lento)
```

Salidas en `analisis/resultados/barrido/<E?>/{results,loss,meta,models,errors}/`.
`results/` es la tabla cruda; `models/` guarda los checkpoints por época (los usa E5).

## 3. OOD / profesor (Parte E)

Instrumentación del **mecanismo de fallo** sobre el politopo del profesor (`n_x=2`, `N=2`):
peor autovalor de lazo cerrado y residual de DR **vs iteración k**, para ŷ de la **red** vs ŷ
**aleatorio** (mediana de varias semillas), más `λ_min(Q)`, `κ(Q)`, factibilidad CVXPY y
tiempos vs `δ`. Entrenamos un modelo `n_x=2` rápido solo para la demo (las etapas del barrido
lo entrenan en serio; E5 evalúa esos modelos sin reentrenar).

**Hipótesis:** con pérdidas que premian `Q` pequeña, ŷ apunta OOD a una esquina casi-singular
donde C1 y C2 se cortan casi tangencialmente → DR converge lentísimo → `K=YQ⁻¹` explota a
presupuesto corto. Predicción distintiva: el **aplanamiento del residual vs k** aparece SOLO
con ŷ de la red, no con ŷ aleatorio.

In [ ]:
from analisis import benchmark as bm
from barrido.ood_eval import professor_mechanism
from barrido.ood_banks import professor_bank

# modelo n_x=2 rápido (demo). cfg_cols solo etiqueta las filas crudas.
res_n2 = bm.run_experiment(arch="actuadores", n=2, N_list=[2, 3], m=1, loss="control",
                           dr_train=30, dr_eval=1000, epochs=40, limit=150,
                           compare_cvxpy=False, verbose=False)
model_n2 = res_n2["model"]
cfg_cols = {"loss": "control", "arch": "actuadores", "n_x": 2, "seed": 42}

deltas  = np.linspace(0.0, 2.0, 9)
budgets = [100, 250, 500, 1000, 2000, 4000]
kgrid   = [25, 50, 100, 200, 400, 700, 1000, 1500, 2000, 3000, 4000]
trace, summ = professor_mechanism(model_n2, cfg_cols, deltas=deltas, budgets=budgets,
                                  k_grid=kgrid, n_random=3)
trace = pd.DataFrame(trace); summ = pd.DataFrame(summ)
display(summ[["delta", "iters_min", "estab_en_max", "lam_min_Q_final", "kappa_Q_final",
              "worst_final", "cvxpy_feasible", "t_stab_red_ms", "t_cvxpy_ms"]].round(3))

In [ ]:
# Mecanismo: peor autovalor y residual de DR vs k (ŷ red vs aleatorio) al δ más cercano a 1.0
d0 = float(trace.delta.iloc[(trace.delta - 1.0).abs().values.argmin()])
sub = trace[np.isclose(trace.delta, d0)]
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for src, col in (("red", "#a93226"), ("aleatorio", "#2471a3")):
    s = sub[sub.source == src]
    axs[0].plot(s.groupby("k").worst.median(), color=col, marker=".", label=f"ŷ {src}")
    axs[1].plot(s.groupby("k").residual.median(), color=col, marker=".", label=f"ŷ {src}")
axs[0].axhline(0, ls="--", color="k", alpha=.5)
axs[0].set(xscale="log", xlabel="k (iteración DR)", ylabel=r"peor Re$\,\lambda$ CL",
           title=f"δ={d0:.2f}: lazo cerrado vs k")
axs[1].set(xscale="log", yscale="log", xlabel="k (iteración DR)", ylabel="residual DR",
           title="residual vs k (aplanamiento sólo con ŷ red)")
axs[0].legend(fontsize=8); axs[1].legend(fontsize=8); fig.tight_layout()

### Pole-shift (`n_x=3`) y E5 (evaluación OOD del barrido)

El banco pole-shift (`n_x=3`, `N=1`) se evalúa con un modelo `n_x=3` (invariante a `N`).
Abajo, una demo del ladder `iters_min` vs `s`. **E5 completo** (todos los modelos entrenados
en E1–E4, sin reentrenar) se corre desde terminal:

```bash
python -m barrido.ood_eval --models analisis/resultados/barrido/E1/models \
                           --out    analisis/resultados/barrido/OOD
```

In [ ]:
from barrido.ood_banks import pole_shift_bank
from barrido.ood_eval import evaluate_bank_ladder

res_n3 = bm.run_experiment(arch="actuadores", n=3, N_list=[2, 3, 4], m=1, loss="control",
                           dr_train=30, dr_eval=1000, epochs=30, limit=150,
                           compare_cvxpy=False, verbose=False)
rows_ps = evaluate_bank_ladder(res_n3["model"], pole_shift_bank(),
                               [100, 500, 1000, 2000, 4000], "pole_shift",
                               {"loss": "control", "n_x": 3})
ps = pd.DataFrame(rows_ps)
g = ps.groupby("param").agg(iters_min=("iters_min", "first"),
                            estab_en_max=("estab_en_max", "first"))
display(g.round(1))

## 4. Figuras (Parte F)

Las figuras se generan **desde los CSV/parquet crudos** (no recomputan). Guardamos el
mecanismo de la demo a shards y armamos P1–P5, más las de entrenamiento desde `E0`.
Para el barrido/OOD completos, correr `python -m barrido.figuras --barrido <E?> --ood <OOD>`.

In [ ]:
from barrido import figuras as F
from barrido.run import save_table

OOD_NB  = ROOT / "analisis" / "resultados" / "barrido" / "OOD_nb"
OUT_FIG = ROOT / "analisis" / "resultados" / "barrido" / "figuras_nb"
save_table(trace, OOD_NB / "mecanismo_trace" / "demo")
save_table(summ,  OOD_NB / "mecanismo_summary" / "demo")

# profesor (P1–P5) desde los shards de la demo
F.figP1_itersmin_vs_delta(OOD_NB, OUT_FIG)
F.figP2_trazas(OOD_NB, OUT_FIG)
F.figP3_certificado_vs_delta(OOD_NB, OUT_FIG)
F.figP4_tiempo_vs_delta(OOD_NB, OUT_FIG)
F.figP5_heatmap(OOD_NB, OUT_FIG)
# entrenamiento desde E0
F.fig_curvas_loss(OUT_NB, OUT_FIG)
F.fig_estabilidad(OUT_NB, OUT_FIG)
print("Figuras guardadas en", OUT_FIG)